# Analyse exploratoire des données (langage R)

In [1]:
# Chargement des librairies nécessaires
library(ggplot2)
library(tidyverse)
library(gridExtra)
library(GGally)
library(plotly)
library(corrplot)
library(reshape2)
library(FactoMineR) 
library(factoextra)
library(glmnet) 
library(ggfortify)
library(pROC)
library(ROCR)

── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ lubridate 1.9.5     ✔ tibble    3.3.1
✔ purrr     1.2.1     ✔ tidyr     1.3.2
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : 'gridExtra'


L'objet suivant est masqué depuis 'package:dplyr':

    combine



Attachement du package : 'plotly'


L'objet suivant est masqué depuis 'package:ggplot2':

    last_plot


L'objet suivant est masqué depuis 'package:stats':

    filter


L'objet suivant est masqué depuis 'package:graphics':

    layout


corrplot 0.95 loaded


Attachement du package : 'reshape2'


L'objet suiv

In [2]:
data= read.table("C:\\Users\\Utilisateur\\Desktop\\Projet_Machine_Learning\\healthcare_synthetic_data.csv",sep=",",header=TRUE)
#head(data)
#summary(data)

**Transformation des données**

In [3]:
#on enlève patient_ID ( A EXECUTER 1 SEULE FOIS)
data=data[,-1]

In [4]:
#Changement du type des variables qualitatives en facteurs
data[,"Gender"]= as.factor(data[,"Gender"])
data[,"Smoking_Status"]= as.factor(data[,"Smoking_Status"])
data[,"Alcohol_Consumption"]= as.factor(data[,"Alcohol_Consumption"])
data[,"Physical_Activity_Level"]= as.factor(data[,"Physical_Activity_Level"])
data[,"Family_History"]= as.factor(data[,"Family_History"])
data[,"Stress_Level"]= as.factor(data[,"Stress_Level"])
data[,"Heart_Disease_Risk"]= as.factor(data[,"Heart_Disease_Risk"])
data[,"Sleep_Hours"]= as.factor(data[,"Sleep_Hours"])


In [5]:
#Tranformation (racine carrée) de Weight_kg et BMI pour centrer les densitées
options(repr.plot.width = 10, repr.plot.height = 5)
data[,"SBMI"]=sqrt(data[,"BMI"])
data[,"SWeight"]=sqrt(data[,"Weight_kg"])
#sg4=ggplot(data,aes(x=SBMI)) + geom_density(linewidth=1,col="darkolivegreen") + geom_histogram(alpha=0.6,aes(y=after_stat(density)))
#sg3=ggplot(data,aes(x=SWeight)) + geom_density(linewidth=1,col="darkolivegreen") + geom_histogram(alpha=0.6,aes(y=after_stat(density)))
#grid.arrange(g3,g4,sg3,sg4,ncol=2)

# Prediction de la variable Heart_Disease_Risk

## 1.Division du jeu de données en un échantillon d’apprentissage et un échantillon test

Pourquoi cette étape est-elle nécessaire lorsque nous nous concentrons sur les performances des algorithmes ?

In [6]:
library(caret)

Le chargement a nécessité le package : lattice


Attachement du package : 'caret'


L'objet suivant est masqué depuis 'package:purrr':

    lift




In [7]:
#on enlève les colonnes weight et BMI que l'on a transformé
data_clean=data[,-c(4,5)]
head(data_clean)

,Age,Gender,Height_cm,Systolic_BP,Diastolic_BP,Cholesterol_Total,Cholesterol_LDL,Cholesterol_HDL,Fasting_Blood_Sugar,Smoking_Status,Alcohol_Consumption,Physical_Activity_Level,Family_History,Stress_Level,Sleep_Hours,Heart_Disease_Risk,SBMI,SWeight
,<int>,<fct>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<dbl>,<dbl>
1,60,0,146.9,140,89,217,151,52,83,0,1,3,0,1,8,0,4.878524,7.162402
2,53,0,161.8,128,81,203,119,38,116,0,0,1,0,7,9,0,5.412947,8.752143
3,62,1,174.7,141,100,173,124,45,90,0,0,0,0,1,7,1,5.504544,9.612492
4,73,1,173.3,136,96,193,117,45,81,0,0,1,0,2,7,1,4.785394,8.300602
5,52,1,178.6,122,80,236,153,41,79,0,1,2,0,2,6,0,5.000000,8.933085
6,52,0,159.6,134,92,225,155,48,103,0,0,1,1,4,8,0,4.868265,7.765307


In [8]:
set.seed(131)
trainIndex = createDataPartition(data_clean$Heart_Disease_Risk,p=0.8,list = FALSE)
dataTrain = data_clean[ trainIndex,]
dataTest = data_clean[-trainIndex,]

## 2. Modèle de régression logistique

Heart_Disease_Risk est une variable binaire, on utilise donc un modèle linéaire généralisé ( régression logistique)

### Modèle sans intéraction entre les variables

In [9]:
# estimation du modèle complet(sans interactions)
reglog = glm(Heart_Disease_Risk ~. , data = dataTrain, family = binomial(link="logit"))
# significativité des paramètres
anova(reglog, test = "Chisq")

,Df,Deviance,Resid. Df,Resid. Dev,Pr(>Chi)
,<int>,<dbl>,<int>,<dbl>,<dbl>
NULL,NA,NA,11999,16438.38,NA
Age,1,3.330504e+02,11998,16105.33,2.081636e-74
Gender,1,5.432754e-01,11997,16104.79,4.610785e-01
Height_cm,1,6.755591e-01,11996,16104.11,4.111201e-01
Systolic_BP,1,2.270577e+02,11995,15877.06,2.612348e-51
Diastolic_BP,1,2.018179e-01,11994,15876.86,6.532575e-01
Cholesterol_Total,1,5.748988e+01,11993,15819.37,3.397228e-14
Cholesterol_LDL,1,1.919658e+01,11992,15800.17,1.179244e-05
Cholesterol_HDL,1,3.190611e+01,11991,15768.26,1.618070e-08


Il est pertinent de se demander si on peut simplifier le modèle en selectionnant seulement les variables significatives. On utilise d'abord le critère AIC (comparaison avec BIC ??)

In [11]:
# Recherche d'un modèle optimal au sens d'Akaïke
reglog.stepAIC = step(reglog, direction = "backward") # éventuellement tester avec direction= "both"

Start:  AIC=12734.3
Heart_Disease_Risk ~ Age + Gender + Height_cm + Systolic_BP + 
    Diastolic_BP + Cholesterol_Total + Cholesterol_LDL + Cholesterol_HDL + 
    Fasting_Blood_Sugar + Smoking_Status + Alcohol_Consumption + 
    Physical_Activity_Level + Family_History + Stress_Level + 
    Sleep_Hours + SBMI + SWeight

                          Df Deviance   AIC
- SWeight                  1    12666 12732
- Height_cm                1    12666 12732
- Diastolic_BP             1    12666 12732
- Cholesterol_Total        1    12667 12733
- SBMI                     1    12667 12733
- Gender                   1    12667 12733
- Alcohol_Consumption      2    12669 12733
<none>                          12666 12734
- Cholesterol_HDL          1    12672 12738
- Sleep_Hours              6    12695 12751
- Cholesterol_LDL          1    12693 12759
- Age                      1    12741 12807
- Systolic_BP              1    12744 12810
- Stress_Level             9    12863 12913
- Fasting_Blood_Su

In [12]:
anova(reglog.stepAIC,test="Chisq")

,Df,Deviance,Resid. Df,Resid. Dev,Pr(>Chi)
,<int>,<dbl>,<int>,<dbl>,<dbl>
NULL,NA,NA,11999,16438.38,NA
Age,1,333.05045,11998,16105.33,2.081636e-74
Systolic_BP,1,227.16047,11997,15878.17,2.480927e-51
Cholesterol_LDL,1,75.68844,11996,15802.48,3.321459e-18
Cholesterol_HDL,1,30.50806,11995,15771.98,3.324821e-08
Fasting_Blood_Sugar,1,199.17945,11994,15572.80,3.154255e-45
Smoking_Status,1,1631.69029,11993,13941.11,0.000000e+00
Physical_Activity_Level,3,396.37344,11990,13544.73,1.351091e-85
Family_History,1,603.82807,11989,12940.91,2.461315e-133


In [13]:
#matrice de confusion
table(reglog$fitted.values > 0.5, dataTrain[, "Heart_Disease_Risk"])
table(reglog.stepAIC$fitted.values > 0.5, dataTrain[, "Heart_Disease_Risk"])

       
           0    1
  FALSE 5494 1851
  TRUE  1274 3381

       
           0    1
  FALSE 5492 1848
  TRUE  1276 3384

### Modèle avec intéractions

Nous travaillons avec un grand nombre de variables et donc d'interactions ce qui peut donner des warnings et être long si on essaye d'estimer le modèle avec toutes les interactions possibles d'un coup. Nous allons donc utiliser deux méthodes qui permettent d'éviter le problème en selectionnant directement les variables et intéractions siginificatives:

- Une procédure stepwise de selection de variable sur critère AIC
- Une pénalisation L1 (LASSO)

In [14]:
#première méthode
# estimation du modèle avec interactions en partant du modèle constant
reglog.inter = glm(Heart_Disease_Risk ~ 1, data = dataTrain,family = binomial)
# algorithme stepwise en précisant le plus grand 
# modèle possible
reglog.inter.step = step(reglog.inter, direction = "both",
    scope = list(lower = ~1, upper = ~(Age + Gender + Height_cm + SWeight + SBMI + Systolic_BP + Diastolic_BP +Cholesterol_Total + Cholesterol_LDL
                                      + Cholesterol_HDL +  Fasting_Blood_Sugar + Smoking_Status +  Alcohol_Consumption + Physical_Activity_Level + 
                                       Family_History + Stress_Level + Sleep_Hours)^2))

Start:  AIC=16440.38
Heart_Disease_Risk ~ 1

                          Df Deviance   AIC
+ Smoking_Status           1    14896 14900
+ Family_History           1    15986 15990
+ Systolic_BP              1    15995 15999
+ Age                      1    16105 16109
+ Fasting_Blood_Sugar      1    16121 16125
+ Physical_Activity_Level  3    16129 16137
+ Diastolic_BP             1    16158 16162
+ Cholesterol_Total        1    16216 16220
+ Cholesterol_LDL          1    16216 16220
+ SBMI                     1    16246 16250
+ SWeight                  1    16299 16303
+ Stress_Level             9    16292 16312
+ Cholesterol_HDL          1    16386 16390
+ Sleep_Hours              6    16418 16432
<none>                          16438 16440
+ Height_cm                1    16437 16441
+ Gender                   1    16438 16442
+ Alcohol_Consumption      2    16436 16442

Step:  AIC=14900.3
Heart_Disease_Risk ~ Smoking_Status

                          Df Deviance   AIC
+ Family_History  

In [15]:
anova(reglog.inter.step, test = "Chisq")

,Df,Deviance,Resid. Df,Resid. Dev,Pr(>Chi)
,<int>,<dbl>,<int>,<dbl>,<dbl>
NULL,NA,NA,11999,16438.38,NA
Smoking_Status,1,1542.083940,11998,14896.30,0.000000e+00
Family_History,1,532.760681,11997,14363.54,7.085177e-118
Systolic_BP,1,497.254186,11996,13866.29,3.761899e-110
Physical_Activity_Level,3,405.425392,11993,13460.86,1.478964e-87
Fasting_Blood_Sugar,1,299.468484,11992,13161.39,4.301016e-67
Stress_Level,9,190.586064,11983,12970.81,3.103894e-36
Cholesterol_LDL,1,145.850768,11982,12824.95,1.399476e-33
Age,1,58.182758,11981,12766.77,2.388634e-14


In [16]:
#Deuxième méthode
x.mat = model.matrix(Heart_Disease_Risk~(.)^2-1, data=dataTrain) # permet d'encoder les variables qualitatives

#Pour illustrer le fonctionnement de la pénalisation, l'evolution des coefficients selon la valeur de lambda
#reglog.inter.lasso = glmnet(x=x.mat,dataTrain$Heart_Disease_Risk, family="binomial", alpha=1)
#options(repr.plot.width = 12, repr.plot.height = 10)
#plot(reglog.inter.lasso, xvar = "lambda", label = TRUE)

On determine la valeur optimale du paramètre $\lambda$, associé a la pénalisation L1, par validation croisée.

In [ ]:
y_train = as.numeric(as.character(dataTrain$Heart_Disease_Risk)) #la fonction cv.glmnet ne prend pas d'objets factors
set.seed(123) #Pour que le découpage des k folds soit toujours le même
reglog.inter.lasso.cv = cv.glmnet(y=y_train, x = x.mat,family="binomial", alpha=1)
autoplot(reglog.inter.lasso.cv)

La ligne pointillée de gauche correspond à la valeur de lambda.min qui donne l'erreur de validation croisée la plus faible. La ligne de pointillé de droite correspond à la valeur de lambda.1se qui donne la plus grande valeur de lambda telle que l'erreur reste à moins d'un écart-type de l'erreur minimale. Cette valeur de lambda donne donc une plus grande pénalisation et donc un modèle plus simple, c'est elle que nous allons utiliser. 

In [ ]:
# valeur estimée
paste("Valeur de lambda estimée :", round(reglog.inter.lasso.cv$lambda.1se, 3)) #round(,3) correspond au nb de décimale après la virgule
# coefficients du modèle correspondant
coef(reglog.inter.lasso.cv, s = "lambda.1se")

Nous allons comparer les variables retenues par chaque modèle:

In [ ]:
# Variables du modèle par stepAIC
vars_step = names(coef(reglog.inter.step))[-1] # [-1] pour enlever l'intercept

# Variables du modèle lasso (en retenant lamda1se)
# On récupère la matrice des coefficients
coefs_lasso = coef(reglog.inter.lasso.cv, s = "lambda.1se")
# On garde les noms dont la valeur est différente de 0
vars_lasso = rownames(coefs_lasso)[which(coefs_lasso != 0)][-1] # [-1] pour l'intercept

#Variables communes aux deux modèles
communes = intersect(vars_step, vars_lasso)

# Variables le StepAIC a-t-il gardées que le LASSO a supprimées 
step = setdiff(vars_step, vars_lasso)

# Variables le LASSO a-t-il gardées que le StepAIC a supprimées 
lasso = setdiff(vars_lasso, vars_step)

paste("Nombre de variables communes:",length(communes))
paste("Nombre de variables uniquement dans le modèle stepAIC:",length(step))
paste("Nombre de variables uniquement dans le modèle LASSO:",length(lasso))

Les deux modèles obtenus sont complètements différents (seulement deux variables communes). Cela peut s'expliquer par le fait que reglog.inter.step considère les variables qualitatives comme des "blocs"alors que reglog.inter.lasso.cv considère chaque modalité.

In [ ]:
#Comparaisons matrices de confusions
table(reglog.inter.step$fitted.values > 0.5, dataTrain[, "Heart_Disease_Risk"])
proba_lasso = predict(reglog.inter.lasso.cv, newx = x.mat, type = "response")  
table(proba_lasso > 0.5,dataTrain$Heart_Disease_Risk)

### Prévision de l'échantillon test

**Matrices de confusions**

In [ ]:
#Prévision modèle sans interaction sans selection de variable
pred.log <- predict(reglog, newdata = dataTest, type = "response")
#Prévision modèle sans interaction avec selection de variable
pred.log.step <- predict(reglog.stepAIC, newdata = dataTest, type = "response")
#Prévision modèle avec interaction par stepAIC
pred.log.inter.step <- predict(reglog.inter.step, newdata = dataTest, type = "response")
#Prévision modèle avec interaction par lasso
x.mat.test = model.matrix(Heart_Disease_Risk~(.)^2-1, data=dataTest)
pred.log.inter.lasso <- predict(reglog.inter.lasso.cv, newx = x.mat.test, type = "response")

# Matrices de confusion
print("Matrice de confusion du modèle reglog :")
table(pred.log > 0.5, dataTest[, "Heart_Disease_Risk"])
print("Matrice de confusion du modèle reglog.stepAIC :")
table(pred.log.step > 0.5, dataTest[, "Heart_Disease_Risk"])
print("Matrice de confusion du modèle reglog.inter.step :")
table(pred.log.inter.step > 0.5, dataTest[, "Heart_Disease_Risk"])
print("Matrice de confusion du modèle reglog.inter.lasso.cv :")
table(pred.log.inter.lasso> 0.5, dataTest[, "Heart_Disease_Risk"])

**Courbe ROC**

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 6)
par(mfrow = c(1, 1))

# modèle sans interaction sans selection de variable
predlogit <- prediction(pred.log, dataTest[, "Heart_Disease_Risk"])
perflogit <- performance(predlogit, "tpr", "fpr")

#Prévision modèle sans interaction avec selection de variable
predlogit.step <- prediction(pred.log.step, dataTest[, "Heart_Disease_Risk"])
perflogit.step <- performance(predlogit.step, "tpr", "fpr")

#Prévision modèle avec interaction par stepAIC
predlogit.interstep <- prediction(pred.log.inter.step, dataTest[, "Heart_Disease_Risk"])
perflogit.inter.step <- performance(predlogit.interstep, "tpr", "fpr")

#Prévision modèle avec interaction par lasso
predlogit.inter.lasso <- prediction(pred.log.inter.lasso, dataTest[, "Heart_Disease_Risk"])
perflogit.inter.lasso <- performance(predlogit.inter.lasso, "tpr", "fpr")

plot(perflogit, col = "blue",lty=2, main = "Courbe ROC ")
plot(perflogit.step,col="orange",lty=2,add=TRUE)
plot(perflogit.inter.step,col="green",lty=1,add=TRUE) 
plot(perflogit.inter.lasso,col="red",lty=1,add=TRUE) 

legend("right", legend=c("reglog", "reglog.stepAIC", "reglog.inter.step","reglog.inter.lasso.cv"),
       col=c("blue","orange","green","red"), lty=c(2,2,1,1), text.font=1,    cex=0.8)


## Modèle SVM

In [11]:
library(e1071)


Attachement du package : 'e1071'


L'objet suivant est masqué depuis 'package:ggplot2':

    element




Justifier le type de noyau utilisé. On part du principe que les données ne sont pas linéairement séparables (très peu probable).

In [12]:
# optimisation
# cost est paramètre qui permet d'ajuster la tolérance aux points mal classées. Plus il est grand, plus il y en a. 
# gamma permet de régler l'influence d'un point. Plus il est élevé, plus l'influence d'un point est courte.
# kernel par défaut (radial)
svm.tune = tune.svm(Heart_Disease_Risk ~ ., data = dataTrain, cost = c(1,1.25,1.5,1.75,2), 
    gamma = seq(0.02, 0.1, by = 0.02))
plot(svm.dis.tune)

ERROR: Error: objet 'dataTrain' introuvable


In [ ]:
# apprentissage
svm.tune$best.parameters
svm.best=svm(Heart_Disease_Risk~.,data=dataTrain,cost = svm.tune$best.parameters$cost, 
    gamma = svm.tune$best.parameters$gamma)